# Benchmark Comparison: 1D-CNN vs DNABERT-2
**Genomic-RawSeq-Analyzer — CMPE 492**

Produces side-by-side ROC curves and the full benchmark table.

**Outputs:**
-
-

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, glob, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.metrics import roc_curve, auc, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

!pip install -q scikit-learn scipy matplotlib seaborn tensorflow

BASE      = '/content/drive/MyDrive/DNA_Anomaly_Detection'
BATCH_DIR = f'{BASE}/BreastCancer_Data_Parts'
MODEL_PATH= f'{BASE}/ML Models/BreastCancer_CNN_Model.keras'
OUT_DIR   = f'{BASE}/results/comparison'
os.makedirs(OUT_DIR, exist_ok=True)
print('Setup OK')
print('Batch files found:', sorted(glob.glob(f'{BATCH_DIR}/batch_*.npz')))

Mounted at /content/drive
Setup OK
Batch files found: ['/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_1.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_2.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_3.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_4.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_5.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_6.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_7.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_8.npz', '/content/drive/MyDrive/DNA_Anomaly_Detection/BreastCancer_Data_Parts/batch_9.npz']


## Step 1 — Load Data & CNN Evaluation

In [3]:
# ── Load all .npz batches ──────────────────────────────────────────────
file_list = sorted(glob.glob(f'{BATCH_DIR}/batch_*.npz'))
if not file_list:
    raise FileNotFoundError(f'No batch_*.npz files in {BATCH_DIR}')

X_list, y_list, id_list = [], [], []
for path in file_list:
    print(f'Loading {os.path.basename(path)}...')
    with np.load(path) as d:
        X_list.append(d['X'])
        y_list.append(d['y'])
        # run_ids are optional — fall back to batch index if not present
        if 'run_ids' in d:
            id_list.append(d['run_ids'])
        else:
            batch_name = os.path.basename(path).replace('.npz','')
            id_list.append(np.array([f'{batch_name}_read_{i}' for i in range(len(d['X']))]))

X       = np.concatenate(X_list)
y       = np.concatenate(y_list)
run_ids = np.concatenate(id_list)

print(f'''
Total loaded: X={X.shape}  y={y.shape}''')
print(f'Tumor={int(y.sum()):,}  Normal={int((y==0).sum()):,}')

# 80/20 split (same seed as training)
idx = np.arange(len(X))
_, test_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)
X_test      = X[test_idx]
y_test      = y[test_idx]
run_ids_test= run_ids[test_idx]
print(f'Test set: {len(X_test):,} reads')

Loading batch_1.npz...
Loading batch_2.npz...
Loading batch_3.npz...
Loading batch_4.npz...
Loading batch_5.npz...
Loading batch_6.npz...
Loading batch_7.npz...
Loading batch_8.npz...
Loading batch_9.npz...

Total loaded: X=(1800000, 80)  y=(1800000,)
Tumor=1,100,000  Normal=700,000
Test set: 360,000 reads


In [4]:
# ── Load CNN and run inference ─────────────────────────────────────────
from tensorflow.keras.models import load_model

cnn_model = load_model(MODEL_PATH)
print('Model loaded:', MODEL_PATH)

t0 = time.perf_counter()
probs_cnn = cnn_model.predict(X_test, batch_size=2048, verbose=1).flatten()
elapsed   = time.perf_counter() - t0
n_batches = max(1, len(X_test) // 2048)
cnn_ms_per_batch = (elapsed / n_batches) * 1000

# Read-level metrics
fpr_cnn_read, tpr_cnn_read, _ = roc_curve(y_test, probs_cnn)
auc_cnn_read = auc(fpr_cnn_read, tpr_cnn_read)
prec_cnn, rec_cnn, f1_cnn, _ = precision_recall_fscore_support(
    y_test, (probs_cnn >= 0.5).astype(int), average='binary', zero_division=0)

# Patient-level crowd-voting
df_cnn  = pd.DataFrame({'run_id': run_ids_test, 'prob': probs_cnn, 'label': y_test})
pat_cnn = df_cnn.groupby('run_id').agg(
    patient_prob =('prob',  'mean'),
    patient_label=('label', lambda x: int(x.mode()[0]))
).reset_index()
fpr_cnn_pat, tpr_cnn_pat, _ = roc_curve(pat_cnn['patient_label'], pat_cnn['patient_prob'])
auc_cnn_pat = auc(fpr_cnn_pat, tpr_cnn_pat)

print(f'CNN Read-level  AUC : {auc_cnn_read:.4f}')
print(f'CNN Patient-level AUC: {auc_cnn_pat:.4f}')
print(f'CNN Inference speed : {cnn_ms_per_batch:.1f} ms/batch')

Model loaded: /content/drive/MyDrive/DNA_Anomaly_Detection/ML Models/BreastCancer_CNN_Model.keras
176/176 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step
CNN Read-level  AUC : 0.6155
CNN Patient-level AUC: 0.6155
CNN Inference speed : 22.6 ms/batch


## Step 2 — DNABERT-2 Pre-computed Results

In [5]:
# ── Pre-computed DNABERT-2 results (from EvalDNABERT2.ipynb) ──────────
DB2_NORMAL_SCORES = np.array([0.581810, 0.580134, 0.575185, 0.543895])
DB2_TUMOR_SCORES  = np.array([0.635969, 0.638958, 0.638554, 0.634829, 0.631615, 0.632352])
DB2_OPTIMAL_THR   = 0.6316
DB2_READ_AUC      = 0.6240
DB2_PRECISION     = 0.6381
DB2_RECALL        = 0.9517
DB2_F1            = 0.7639
DB2_TRAIN_MIN     = 32.0
DB2_INFER_MS      = 205.9
DB2_GPU_MB        = 5816
DB2_PARAMS        = 117_000_000
CNN_TRAIN_MIN     = 5.0
CNN_GPU_MB        = 2000
CNN_PARAMS        = 100_000

# Patient-level ROC from actual scores
labels_db = np.concatenate([np.zeros(len(DB2_NORMAL_SCORES)), np.ones(len(DB2_TUMOR_SCORES))])
scores_db = np.concatenate([DB2_NORMAL_SCORES, DB2_TUMOR_SCORES])
fpr_db_pat, tpr_db_pat, _ = roc_curve(labels_db, scores_db)
auc_db_pat = auc(fpr_db_pat, tpr_db_pat)

# Read-level ROC reconstructed via equal-variance normal model
d_prime     = np.sqrt(2.0) * norm.ppf(DB2_READ_AUC)
fpr_db_read = np.linspace(0, 1, 500)
tpr_db_read = norm.cdf(norm.ppf(np.clip(fpr_db_read, 1e-7, 1-1e-7)) + d_prime)
tpr_db_read[0], tpr_db_read[-1] = 0.0, 1.0

print(f'DNABERT-2 Read-level AUC   : {DB2_READ_AUC:.4f}  (pre-computed)')
print(f'DNABERT-2 Patient-level AUC: {auc_db_pat:.4f}')

DNABERT-2 Read-level AUC   : 0.6240  (pre-computed)
DNABERT-2 Patient-level AUC: 1.0000


## Step 3 — ROC Comparison Plot

In [7]:
COLOR_CNN = '#34495e'
COLOR_DB  = '#e74c3c'
COLOR_RND = '#95a5a6'

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), dpi=150)

for ax, (fpr_cnn, tpr_cnn, fpr_db, tpr_db, auc_cnn, title) in zip(axes, [
    (fpr_cnn_read, tpr_cnn_read, fpr_db_read, tpr_db_read, auc_cnn_read, 'Read-Level ROC'),
    (fpr_cnn_pat,  tpr_cnn_pat,  fpr_db_pat,  tpr_db_pat,  auc_cnn_pat,  'Patient-Level ROC (Crowd-Voting)'),
]):
    ax.plot(fpr_cnn, tpr_cnn, color=COLOR_CNN, lw=2.5, label=f'1D-CNN  (AUC = {auc_cnn:.4f})')
    ax.plot(fpr_db,  tpr_db,  color=COLOR_DB,  lw=2.5, label=f'DNABERT-2  (AUC = {(auc_db_pat if 'Patient' in title else DB2_READ_AUC):.4f})')
    ax.plot([0,1],[0,1], color=COLOR_RND, lw=1.2, linestyle='--', label='Random Chance')
    ax.set_xlim([-0.01,1.01]); ax.set_ylim([-0.01,1.01])
    ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Positive Rate',  fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)

plt.tight_layout()
out_path = f'{OUT_DIR}/roc_comparison.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

Saved: /content/drive/MyDrive/DNA_Anomaly_Detection/results/comparison/roc_comparison.png


## Step 4 — Benchmark Tables

In [9]:
pat_cnn['pred'] = (pat_cnn['patient_prob'] >= 0.5).astype(int)
prec_cnn_pat, rec_cnn_pat, f1_cnn_pat, _ = precision_recall_fscore_support(
    pat_cnn['patient_label'], pat_cnn['pred'], average='binary', zero_division=0)

print('=== TABLE 8.2: READ-LEVEL PERFORMANCE ===')
display(pd.DataFrame({
    'Model'    : ['1D-CNN (Baseline)', 'DNABERT-2 (fine-tuned)'],
    'AUC-ROC'  : [round(auc_cnn_read, 4), DB2_READ_AUC],
    'Precision': [round(prec_cnn, 4),     DB2_PRECISION],
    'Recall'   : [round(rec_cnn, 4),      DB2_RECALL],
    'F1 Score' : [round(f1_cnn, 4),       DB2_F1],
}).set_index('Model'))

print('\n=== TABLE 8.3: PATIENT-LEVEL PERFORMANCE ===')
display(pd.DataFrame({
    'Model'    : ['1D-CNN (Baseline)', 'DNABERT-2 (fine-tuned)'],
    'AUC-ROC'  : [round(auc_cnn_pat, 4), 1.0000],
    'Precision': [round(prec_cnn_pat, 4), 1.0000],
    'Recall'   : [round(rec_cnn_pat, 4),  1.0000],
    'F1 Score' : [round(f1_cnn_pat, 4),   1.0000],
    'Threshold': ['0.50', f'{DB2_OPTIMAL_THR:.4f}'],
}).set_index('Model'))

print('\n=== TABLE 8.4: COMPUTATIONAL PERFORMANCE ===')
display(pd.DataFrame({
    'Model'                : ['1D-CNN (Baseline)', 'DNABERT-2 (fine-tuned)'],
    'Train Time/Epoch'     : [f'~{CNN_TRAIN_MIN:.0f} min', f'~{DB2_TRAIN_MIN:.0f} min'],
    'Inference (ms/batch)' : [f'{cnn_ms_per_batch:.1f}', f'{DB2_INFER_MS:.1f}'],
    'GPU Peak Memory'      : [f'<{CNN_GPU_MB//1000} GB', f'{DB2_GPU_MB:,} MB'],
    'Parameters'           : [f'~{CNN_PARAMS//1000}K', f'{DB2_PARAMS//1_000_000}M'],
}).set_index('Model'))

# Save CSV
csv_path = f'{OUT_DIR}/benchmark_table.csv'
pd.DataFrame({
    'Model'              : ['1D-CNN', 'DNABERT-2'],
    'Read AUC'           : [round(auc_cnn_read, 4), DB2_READ_AUC],
    'Read Precision'     : [round(prec_cnn, 4),     DB2_PRECISION],
    'Read Recall'        : [round(rec_cnn, 4),      DB2_RECALL],
    'Read F1'            : [round(f1_cnn, 4),        DB2_F1],
    'Patient AUC'        : [round(auc_cnn_pat, 4),   1.0000],
    'Train min/epoch'    : [CNN_TRAIN_MIN,            DB2_TRAIN_MIN],
    'Inference ms/batch' : [round(cnn_ms_per_batch, 1), DB2_INFER_MS],
    'GPU MB'             : [CNN_GPU_MB,               DB2_GPU_MB],
}).to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

=== TABLE 8.2: READ-LEVEL PERFORMANCE ===


,AUC-ROC,Precision,Recall,F1 Score
Model,,,,
1D-CNN (Baseline),0.6155,0.6408,0.9885,0.7776
DNABERT-2 (fine-tuned),0.6240,0.6381,0.9517,0.7639



=== TABLE 8.3: PATIENT-LEVEL PERFORMANCE ===


,AUC-ROC,Precision,Recall,F1 Score,Threshold
Model,,,,,
1D-CNN (Baseline),0.6155,0.6408,0.9885,0.7776,0.50
DNABERT-2 (fine-tuned),1.0000,1.0000,1.0000,1.0000,0.6316



=== TABLE 8.4: COMPUTATIONAL PERFORMANCE ===


,Train Time/Epoch,Inference (ms/batch),GPU Peak Memory,Parameters
Model,,,,
1D-CNN (Baseline),~5 min,22.6,<2 GB,~100K
DNABERT-2 (fine-tuned),~32 min,205.9,"5,816 MB",117M



Saved: /content/drive/MyDrive/DNA_Anomaly_Detection/results/comparison/benchmark_table.csv
